# Rescore existing poses with the public API

Run this notebook in a Python kernel with BindingMT, MolSysMT and the Vina
backend available. Supply these objects **before** running the code cells:

| Input | Meaning |
| --- | --- |
| `source_result` | An existing scored `dmt.DockingResult`, with unique nonempty pose IDs and verified pose/source maps. |
| `source_prepared` | The corresponding complete, source-backed `dmt.PreparedLigand`, with valid charges and named types. |
| `source_receptor` | The matching prepared receptor or admitted PDBQT input. |
| `source_domain` | The explicitly chosen `dmt.BoxRegion` for the scoring maps. |
| `active_torsion_selections` | A list of MolSysMT selection strings; each selects exactly two endpoints of one intended active bond on each reconstructed model. Use `[]` for an explicitly rigid ligand. |
| `output_directory` | A fresh destination directory. Existing destinations are refused. |

For example, `"atom_name in ['C3', 'C1']"` is appropriate only when those
labels identify the intended bond in your admitted source. Choose the
receptor, state, box and torsions for your system. This notebook supplies no
molecular preparation recipe and cannot recover missing chemistry.

Each pose is reconstructed through MolSysMT and explicitly prepared using its
existing mechanical assignments. The two evaluations, `vina_fixed` and
`vinardo_fixed`, must be unused names. Both protocols request one CPU, seed 7,
input capture and phase timings. No hydrogen generation, charge calculation,
typing calculation, search or optimization is requested.

The original collection is saved in `source_result.json` before molecular
work. Each returned evaluation is saved in `pose_N/vina_fixed.json` or
`pose_N/vinardo_fixed.json` before preservation checks or the next evaluation.
The ledger retains the intended population, started/completed/failed steps and
evaluations not attempted. An unexpected failure stops the loop; remaining
poses stay not attempted. Failure of the second evaluation keeps the first.
The final cell saves the partial collection and audit before raising that
failure again. A source-integrity failure stops immediately; completed
per-pose captures and the ledger remain saved.
The inputs remain unchanged. `prepared_problems` is aligned with the original
poses; a case whose preparation fails keeps `None` in that position.

`updated_result.json` retains every intended pose, its original position and
the latest saved evaluation, including partial failures. It preserves the
original collection provenance, ranks and ranking decisions. Those ranks
still describe their earlier decision. No new comparison or ordering decision,
or collection-stage provenance, is invented. `updated_audit.json` reports the
actual offline audit; missing or contradictory evidence remains visible.
Audits do not certify chemistry or affinity. Source checks compare prepared
declarations, full-precision coordinates and rendered inputs; they are not
an exhaustive native chemistry/MM audit. Input inspections are not a
complete prepared-model/MM persistence format. Scientific receiving of this
workflow is separate from this example.


In [ ]:
import json
from copy import deepcopy
from pathlib import Path

import molsysmt as msm
import pyunitwizard as puw

import bindingmt as dmt

# Use the caller's existing kernel objects; no inputs are constructed here.
source_result: dmt.DockingResult = globals()['source_result']
source_prepared: dmt.PreparedLigand = globals()['source_prepared']
source_receptor: dmt.PreparedReceptor | str | Path = globals()['source_receptor']
source_domain: dmt.BoxRegion = globals()['source_domain']
active_torsion_selections: list[str] = globals()['active_torsion_selections']
output_directory: str | Path = globals()['output_directory']

if not isinstance(source_result, dmt.DockingResult):
    raise TypeError('source_result must be a DockingResult.')
if not isinstance(source_prepared, dmt.PreparedLigand):
    raise TypeError('source_prepared must be a PreparedLigand.')
if source_prepared.source_molsys is None:
    raise ValueError('Supply the complete source-backed preparation.')
if not isinstance(source_domain, dmt.BoxRegion):
    raise TypeError('source_domain must be an explicitly chosen BoxRegion.')
if not isinstance(active_torsion_selections, list) or any(
    not isinstance(selection, str) or not selection.strip()
    for selection in active_torsion_selections
):
    raise TypeError('Supply a list of explicit active-bond selections.')

original_poses = source_result.poses
pose_ids = [pose.pose_id for pose in original_poses]
if not pose_ids or any(
    not isinstance(pose_id, str) or not pose_id.strip() for pose_id in pose_ids
):
    raise ValueError('Supply a nonempty population with nonempty string pose IDs.')
if len(pose_ids) != len(set(pose_ids)):
    raise ValueError('Pose IDs must be unique.')
evaluations = [('vina_fixed', 'vina'), ('vinardo_fixed', 'vinardo')]
for pose in original_poses:
    if any(
        name == label or name.startswith(label + '.')
        for name in pose.scores
        for label, _ in evaluations
    ):
        raise ValueError('Choose inputs without the evaluation names used here.')

destination = Path(output_directory)
if destination.exists():
    raise FileExistsError('Use a fresh output_directory.')


def save_json(path, record):
    text = json.dumps(record, indent=2, allow_nan=False) + '\n'
    path.write_text(text, encoding='utf-8')


def inspect_prepared(prepared):
    return {
        'declaration': deepcopy(prepared.to_dict()),
        'coordinates': puw.convert(prepared.coordinates, to_form='record').to_dict(),
        'pdbqt': prepared.to_pdbqt(),
    }


def inspect_receptor():
    if isinstance(source_receptor, dmt.PreparedReceptor):
        return inspect_prepared(source_receptor)
    if isinstance(source_receptor, Path):
        return source_receptor.read_bytes()
    if isinstance(source_receptor, str):
        if source_receptor.startswith('pdbqt_text:') or '\n' in source_receptor:
            return source_receptor
        return Path(source_receptor).read_bytes()
    raise TypeError('Supply a PreparedReceptor or an admitted PDBQT input.')


source_result_record = source_result.to_dict()
new_records = deepcopy(source_result_record['poses'])
prepared_problems = [None] * len(original_poses)
ledger = {
    'schema_version': '1.0',
    'scope': 'public_fixed_pose_rescoring',
    'cases': [
        {
            'position': index + 1,
            'pose_id': pose_id,
            'steps': {
                stage: 'not_attempted'
                for stage in ('preparation', 'vina_fixed', 'vinardo_fixed')
            },
            'events': [],
        }
        for index, pose_id in enumerate(pose_ids)
    ],
}
destination.mkdir(parents=True, exist_ok=False)
save_json(destination / 'source_result.json', source_result_record)
save_json(destination / 'ledger.json', ledger)
source_prepared_before = inspect_prepared(source_prepared)
source_receptor_before = inspect_receptor()
source_domain_before = deepcopy(source_domain.to_dict())
torsion_selections_before = list(active_torsion_selections)
save_json(destination / 'source_prepared_inspection.json', source_prepared_before)


def record_event(case, stage, status, **details):
    case['steps'][stage] = status
    case['events'].append({'stage': stage, 'status': status, **details})
    save_json(destination / 'ledger.json', ledger)


def check_sources():
    assert source_result.to_dict() == source_result_record
    assert inspect_prepared(source_prepared) == source_prepared_before
    assert inspect_receptor() == source_receptor_before
    assert source_domain.to_dict() == source_domain_before
    assert active_torsion_selections == torsion_selections_before

In [ ]:
failure = None
for index, original_pose in enumerate(original_poses):
    case = ledger['cases'][index]
    case_directory = destination / f'pose_{index + 1}'
    case_directory.mkdir()
    stage = 'preparation'
    try:
        record_event(case, stage, 'started')
        posed = original_pose.to_molecular_system(source_prepared)
        pairs = []
        for selection in active_torsion_selections:
            endpoints = msm.select(posed, selection=selection)
            if len(endpoints) != 2 or len(set(endpoints)) != 2:
                raise ValueError(
                    f'Active-bond selection must identify two atoms: {selection!r}'
                )
            pairs.append(tuple(int(atom) for atom in endpoints))
        prepared = dmt.prepare_ligand(
            posed,
            selection='all',
            state_id=source_prepared.state_id,
            active_torsion_bonds=pairs,
        )
        problem = dmt.DockingProblem(
            receptor=source_receptor,
            partner=prepared,
            search_domain=source_domain,
            metadata={
                'partner_state_id': prepared.state_id,
                'receptor_state_id': original_pose.receptor_state_id,
            },
        )
        prepared_problems[index] = problem
        record_event(case, stage, 'completed')

        stage = 'source_integrity'
        check_sources()
        current = original_pose
        for label, method in evaluations:
            stage = label
            record_event(case, stage, 'started')
            previous = current.to_dict()
            current = dmt.score(
                problem,
                dmt.VinaProtocol(
                    scoring=method,
                    cpu=1,
                    seed=7,
                    capture_backend_inputs=True,
                    collect_timings=True,
                ),
                pose=current,
                score_name=label,
            )
            record = current.to_dict()
            relative_path = case_directory.relative_to(destination) / f'{label}.json'
            save_json(destination / relative_path, record)
            new_records[index] = record
            record_event(case, stage, 'completed', record=str(relative_path))

            # Each completed record is saved before checks and later operations.
            stage = f'{label}_preservation'
            record_event(case, stage, 'started')
            for field in ('pose_id', 'rank', 'partner_state_id', 'receptor_state_id'):
                assert record[field] == previous[field]
            assert record['coordinates'] == previous['coordinates']
            assert all(
                record['scores'][name] == value
                for name, value in previous['scores'].items()
            )
            old_history = previous['metadata'].get('scoring_history', [])
            assert record['metadata']['scoring_history'][:-1] == old_history
            record_event(case, stage, 'completed')
            stage = 'source_integrity'
            check_sources()

    except Exception as error:
        record_event(
            case,
            stage,
            'failed',
            error_type=type(error).__name__,
            reason=str(error),
        )
        if stage == 'source_integrity':
            raise
        failure = error
        break

In [ ]:
# Keep every intended case in its original position, including partial failures.
combined_record = source_result.to_dict()
combined_record['poses'] = new_records
rescored_result = dmt.DockingResult.from_dict(combined_record)
save_json(destination / 'updated_result.json', rescored_result.to_dict())

# Report the actual audit. Missing collection evidence is not filled in here.
result_audit = dmt.audit_result(rescored_result.to_dict())
save_json(destination / 'updated_audit.json', result_audit)

check_sources()
assert rescored_result.provenance == source_result_record['provenance']
assert rescored_result.ranking_history == source_result.ranking_history
assert [pose.pose_id for pose in rescored_result.poses] == pose_ids
assert (
    dmt.DockingResult.from_dict(rescored_result.to_dict()).to_dict()
    == rescored_result.to_dict()
)

print(f'Saved {len(new_records)} intended cases in {destination}')
print(f'Collection audit: {result_audit["status"]}')
for case in ledger['cases']:
    print(case['pose_id'], case['steps'])
if failure is not None:
    raise failure